# Diagnóstico — datos crudos MINEDUC

Working directory del notebook = `DPL/` → rutas a datos con `../`.

Fuente: `../data/dataset.csv` (unión de los 23 CSV en `data/crudo/`, 94,533 registros).

Cada sección: (1) código, (2) output, (3) **tabla + conclusión** en markdown.


## 0. Carga del conjunto unido


In [ ]:
import pandas as pd
import re

df = pd.read_csv("../data/dataset.csv", encoding="utf-8-sig")
df.shape


## 1. Número de registros y variables


In [ ]:
print(f"Registros: {df.shape[0]}")
print(f"Variables: {df.shape[1]}")


### Conclusión 

| Métrica | Valor |
|---|---|
| Registros | 94,533 |
| Variables | 17 |

- Coincide con el total del scraping → la unión no perdió ni duplicó registros.


## 2. Tipo de dato de cada variable

Comparar tipo inferido por pandas vs tipo semántico.


In [ ]:
df.dtypes


In [ ]:
df.info()


In [ ]:
for col in ["CODIGO", "DISTRITO", "TELEFONO", "NIVEL", "SECTOR"]:
    print(col, "→", df[col].dropna().head(3).tolist())


### Conclusión 

Las 17 columnas se inferieron como `str`. Correcto: `CODIGO`/`DISTRITO` tienen guiones y ceros a la izquierda; `TELEFONO` no es magnitud numérica.

| Variable | Tipo inferido | Tipo esperado | ¿Coincide? |
|---|---|---|---|
| CODIGO | str | identificador `##-##-####-##` | Sí |
| DISTRITO | str | identificador textual | Sí |
| DEPARTAMENTO…DEPARTAMENTAL (resto) | str | texto / categórica | Sí |



## 3. Valores faltantes por variable

NaN reales + faltantes disfrazados (`-`, `S/D`, `.`, etc.).


In [ ]:
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(2)
pd.DataFrame({"faltantes": faltantes, "porcentaje": porcentaje}).sort_values("porcentaje", ascending=False)


In [ ]:
TOKENS_FALTANTES = {
    "", "-", "--", "---", ".", "..", "?",
    "N/A", "NA", "n/a", "na", "NULL", "null", "None", "none",
    "S/D", "SD", "SIN DATO", "SIN DATOS", "NO APLICA", "N.A.", "N.A",
}

def mascara_disfrazados(serie: pd.Series) -> pd.Series:
    out = pd.Series(False, index=serie.index)
    no_nulos = serie.notna()
    texto = serie[no_nulos].astype(str).str.strip()
    # también cualquier cadena hecha solo de guiones (----, -----…)
    out.loc[no_nulos] = (
        texto.eq("") | texto.isin(TOKENS_FALTANTES) | texto.str.fullmatch(r"-+", na=False)
    )
    return out

filas = []
for col in df.columns:
    nan_n = int(df[col].isnull().sum())
    mask = mascara_disfrazados(df[col])
    dis_n = int(mask.sum())
    ejemplos = (
        df.loc[mask, col].astype(str).str.strip().value_counts().head(5).to_dict()
        if dis_n else {}
    )
    filas.append({
        "variable": col,
        "faltantes_nan": nan_n,
        "pct_nan": round(nan_n / len(df) * 100, 2),
        "faltantes_disfrazados": dis_n,
        "pct_disfrazados": round(dis_n / len(df) * 100, 2),
        "ejemplos_disfrazados": ejemplos,
    })

resumen_faltantes = pd.DataFrame(filas).sort_values(
    ["pct_nan", "pct_disfrazados"], ascending=False
).reset_index(drop=True)
resumen_faltantes


### Conclusión 

| Variable | # NaN | % | # disfrazados* | % |
|---|---|---|---|---|
| DIRECTOR | 27,739 | 29.34 | ~2,060+ | ~2.2 |
| TELEFONO | 27,514 | 29.11 | 133 (`S/D`) | 0.14 |
| SUPERVISOR | 10,702 | 11.32 | 0 | 0 |
| DISTRITO | 10,694 | 11.31 | 0† | 0 |
| DIRECCION | 863 | 0.91 | 22 | 0.02 |
| ESTABLECIMIENTO | 15 | 0.02 | 0 | 0 |
| Resto (catálogo + CODIGO) | 0 | 0 | 0 | 0 |



## 4. Cantidad de valores únicos

`nunique` por variable y listado completo de categóricas de catálogo para detectar variantes de escritura.


In [ ]:
df.nunique(dropna=True).sort_values(ascending=False)


In [ ]:
CATEGORICAS = [
    "DEPARTAMENTO", "NIVEL", "SECTOR", "AREA", "STATUS",
    "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL",
]

for col in CATEGORICAS:
    vals = sorted(df[col].dropna().astype(str).unique().tolist())
    print(f"\n=== {col} ({len(vals)}) ===")
    for v in vals:
        print(repr(v))


In [ ]:
# Variantes por strip / mayúsculas (¿colapsan categorías?)
for col in CATEGORICAS + ["MUNICIPIO"]:
    s = df[col].dropna().astype(str)
    raw, stripped, upper = s.nunique(), s.str.strip().nunique(), s.str.strip().str.upper().nunique()
    if raw != stripped or stripped != upper:
        print(f"{col}: raw={raw}, strip={stripped}, upper={upper}")
    else:
        print(f"{col}: sin colapso por strip/case ({raw} únicos)")


### Conclusión — §4

| Variable | # únicos | Observación |
|---|---|---|
| CODIGO | 94,533 | Clave candidata: 1 por fila |
| DIRECCION | 44,489 | Texto libre |
| TELEFONO | 36,706 | Texto libre / multi-valor |
| DIRECTOR | 35,947 | Texto libre (+ placeholders) |
| ESTABLECIMIENTO | 22,782 | Texto libre |
| DISTRITO | 2,275 | Identificador (formatos mixtos → §6) |
| SUPERVISOR | 1,639 | Texto libre |
| MUNICIPIO | 357 | Incluye `ZONA 1`…`ZONA 25` (solo CIUDAD CAPITAL) |
| DEPARTAMENTAL | 26 | Direcciones departamentales MINEDUC (con tildes) |
| DEPARTAMENTO | 23 | 22 deptos + `CIUDAD CAPITAL` |
| PLAN | 13 | Sin variantes de case; hay subtipos SEMIPRESENCIAL |
| NIVEL | 10 | Incluye `UNIVERSIDAD` (1) y `ADMINISTRATIVOS` (2) |
| STATUS | 6 | |
| JORNADA | 6 | Incluye `SIN JORNADA` |
| SECTOR | 4 | OFICIAL/PRIVADO/MUNICIPAL/COOPERATIVA |
| AREA | 3 | Incluye `SIN ESPECIFICAR` (8 filas) |
| MODALIDAD | 2 | |



## 5. Registros duplicados


In [ ]:
duplicados_exactos = int(df.duplicated().sum())
duplicados_por_codigo = int(df.duplicated(subset=["CODIGO"]).sum())
print(f"Duplicados exactos (toda la fila): {duplicados_exactos}")
print(f"Duplicados por CODIGO: {duplicados_por_codigo}")
print(f"CODIGO nunique={df['CODIGO'].nunique()} | filas={len(df)}")


In [ ]:
# Mismo nombre+dirección con distintos códigos (no es duplicado de fila,
# pero sí señal de sedes/jornadas/niveles distintos o datos repetidos a nivel textual)
g = (
    df.dropna(subset=["ESTABLECIMIENTO", "DIRECCION"])
      .groupby(["ESTABLECIMIENTO", "DIRECCION"])["CODIGO"]
      .nunique()
)
print(f"Pares nombre+dirección con >1 CODIGO: {(g > 1).sum()}")
print(f"Máximo códigos por mismo nombre+dirección: {g.max()}")


### Conclusión 

| Chequeo | Resultado |
|---|---|
| Duplicados exactos de fila | **0** |
| Duplicados por `CODIGO` | **0** (`CODIGO` es único en las 94,533 filas) |
| Mismo `ESTABLECIMIENTO`+`DIRECCION`, distinto `CODIGO` | 9,530 pares |

`CODIGO` funciona como llave primaria del crudo. Los 9,530 pares nombre+dirección compartidos **no** son errores por sí solos (mismo plantel puede tener varios códigos por nivel/jornada/plan); son candidatos a revisar en análisis, no a borrar en limpieza automática.


## 6. Valores fuera de dominio o inconsistentes

Dominio de `DEPARTAMENTO` = los 23 values del dropdown de scraping.  
También: formato de `CODIGO`/`DISTRITO`, y coherencia `DEPARTAMENTO` ↔ `DEPARTAMENTAL`.


In [ ]:
# Dominio esperado = exactamente los 23 values scrapeados
DOMINIO_DEPARTAMENTO = {
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA",
}
encontrados = set(df["DEPARTAMENTO"].dropna().astype(str).unique())
print("fuera de dominio DEPARTAMENTO:", sorted(encontrados - DOMINIO_DEPARTAMENTO))
print("faltan respecto al dominio:", sorted(DOMINIO_DEPARTAMENTO - encontrados))


In [ ]:
# Dominios observados de catálogo (para documentar; no hay lista oficial aparte en el CSV)
for col in ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]:
    print(col, "→", sorted(df[col].dropna().astype(str).unique().tolist()))


In [ ]:
# Formato CODIGO / DISTRITO
pat_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
pat_dist_a = re.compile(r"^\d{2}-\d{2}-\d{4}$")  # AA-BB-CCCC
pat_dist_b = re.compile(r"^\d{2}-\d{3}$")         # AA-BBB

codigo_ok = df["CODIGO"].astype(str).str.fullmatch(pat_codigo.pattern)
print(f"CODIGO formato ##-##-####-##: {codigo_ok.sum()} ok | {(~codigo_ok).sum()} fuera")

dist = df["DISTRITO"].dropna().astype(str)
tipo_a = dist.str.fullmatch(pat_dist_a.pattern)
tipo_b = dist.str.fullmatch(pat_dist_b.pattern)
otro = ~(tipo_a | tipo_b)
print(f"DISTRITO AA-BB-CCCC: {tipo_a.sum()}")
print(f"DISTRITO AA-BBB:     {tipo_b.sum()}")
print(f"DISTRITO otro/incompleto: {otro.sum()}")
print(dist[otro].value_counts().head(10))


In [ ]:
# DEPARTAMENTO vs DEPARTAMENTAL (cardinalidad)
print(df.groupby("DEPARTAMENTO")["DEPARTAMENTAL"].nunique().sort_values(ascending=False).head(8))
print("\nCIUDAD CAPITAL →", sorted(df.loc[df["DEPARTAMENTO"]=="CIUDAD CAPITAL", "DEPARTAMENTAL"].unique()))
print("GUATEMALA →", sorted(df.loc[df["DEPARTAMENTO"]=="GUATEMALA", "DEPARTAMENTAL"].unique()))
print("QUICHE →", df.loc[df["DEPARTAMENTO"]=="QUICHE", "DEPARTAMENTAL"].value_counts().to_dict())


### Conclusión 

| Variable | Dominio / regla | Fuera de dominio / inconsistencia | # afectados |
|---|---|---|---|
| DEPARTAMENTO | 23 values del dropdown MINEDUC | Ninguno | 0 |
| NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN | Conjuntos cerrados observados | Sin basura; valores raros pero válidos en catálogo (`UNIVERSIDAD`=1, `ADMINISTRATIVOS`=2, `AREA=SIN ESPECIFICAR`=8) | — |
| CODIGO | `##-##-####-##` | Ninguno | 0 |
| DISTRITO | idealmente `##-##-####` | Formato mixto: 54,911 `AA-BB-CCCC` + 28,849 `AA-BBB` + **79 incompletos** (`01-`, `17-`…) | 79 incompletos |
| DEPARTAMENTAL | dirección administrativa MINEDUC | No es 1:1 con DEPARTAMENTO: CIUDAD CAPITAL y GUATEMALA comparten 4 (`GUATEMALA NORTE/SUR/ORIENTE/OCCIDENTE`); QUICHE → `QUICHÉ` + `QUICHÉ NORTE` | esperado por diseño MINEDUC |
| MUNICIPIO | municipios + zonas | `ZONA *` solo bajo `CIUDAD CAPITAL` (7,594 filas) — coherente con el dropdown | 0 anomalías |

**Nota de enunciado:** aparecen `UNIVERSIDAD` y `ADMINISTRATIVOS` aunque el enfoque era “hasta diversificado”; están en el crudo porque se consultó Nivel=`TODOS`. Decidir en limpieza si se filtran.

## 7. Formatos inconsistentes

Espacios, mayúsculas, formato de teléfono, espacios dobles, encoding.


In [ ]:
# Espacios al inicio/final
for col in df.columns:
    s = df[col].dropna().astype(str)
    n_strip = int((s != s.str.strip()).sum())
    if n_strip:
        print(f"{col}: necesitan strip → {n_strip}")
if all((df[c].dropna().astype(str) == df[c].dropna().astype(str).str.strip()).all() for c in df.columns):
    print("Ninguna columna tiene espacios leading/trailing en valores no nulos.")


In [ ]:
# Mayúsculas
for col in ["ESTABLECIMIENTO", "DIRECCION", "DIRECTOR", "SUPERVISOR", "MUNICIPIO"]:
    s = df[col].dropna().astype(str).str.strip()
    s = s[~s.str.fullmatch(r"-+") & ~s.isin(["S/D", ".", "--", "---"])]
    print(f"{col}: %isupper={s.str.isupper().mean()*100:.1f} | n={len(s)}")


In [ ]:
# Espacios dobles en nombres/direcciones
for col in ["ESTABLECIMIENTO", "DIRECCION"]:
    s = df[col].dropna().astype(str)
    n = int(s.str.contains(r"  +", regex=True).sum())
    print(f"{col}: con espacios dobles (o más) → {n}")
    if n:
        print("  ej:", s[s.str.contains(r"  +", regex=True)].head(3).tolist())


In [ ]:
# Teléfonos: longitud y multi-valor
tel = df["TELEFONO"].dropna().astype(str).str.strip()
tel = tel[~tel.isin(["S/D"]) & ~tel.str.fullmatch(r"-+")]
print("longitudes más frecuentes:")
print(tel.str.len().value_counts().head(8).to_string())
multi = tel.str.contains(r"[,/;]|\sY\s", regex=True)
print(f"\nPosibles multi-teléfono (/,;, Y): {multi.sum()}")
print("ej:", tel[multi].head(5).tolist())


In [ ]:
# Encoding roto (mojibake)
pat_mojibake = r"Ã.|Â.|�"
for col in ["ESTABLECIMIENTO", "DIRECCION", "DIRECTOR", "SUPERVISOR", "DEPARTAMENTAL"]:
    n = int(df[col].dropna().astype(str).str.contains(pat_mojibake, regex=True, na=False).sum())
    print(f"{col}: mojibake-ish → {n}")


### Conclusión 

| Variable | Problema de formato | # afectados | Ejemplo |
|---|---|---|---|
| (todas) | Espacios leading/trailing | **0** | — |
| ESTABLECIMIENTO | Espacios internos dobles | 4,676 | `'EODP NO.21  GABRIELA MISTRAL'` |
| DIRECCION | Espacios internos dobles (si aplica) | ver output | — |
| ESTABLECIMIENTO / MUNICIPIO / SUPERVISOR | Casi 100% MAYÚSCULAS | mayoría | consistente |
| DIRECTOR | Placeholders de guiones de largo variable | ~2,060 | `'----'`, `'----------'` |
| TELEFONO | Multi-valor en una celda | 268 | `'232-5763, 232-6725…'` |
| TELEFONO | Longitud dominante 8 dígitos; otros formatos | — | con guiones / “Y” |
| CODIGO | Formato homogéneo | 0 problemas | `00-01-0001-42` |
| DISTRITO | Dos esquemas + incompletos | 28,849 + 79 | `01-01-0030` vs `01-403` vs `01-` |
| DEPARTAMENTAL vs DEPARTAMENTO | Tildes en uno sí y en otro no | sistemático | `QUICHE` vs `QUICHÉ` |
| Texto | Mojibake (`Ã`, ``) | **0** | encoding OK con utf-8-sig |



## 8. Síntesis — problemas potenciales de calidad


In [ ]:
# Co-ausencia DISTRITO / SUPERVISOR
both = df["DISTRITO"].isna() & df["SUPERVISOR"].isna()
print(f"NaN en ambos DISTRITO y SUPERVISOR: {both.sum()}")
print(f"NaN solo DISTRITO: {(df['DISTRITO'].isna() & df['SUPERVISOR'].notna()).sum()}")
print(f"NaN solo SUPERVISOR: {(df['SUPERVISOR'].isna() & df['DISTRITO'].notna()).sum()}")

# Niveles fuera del foco "hasta diversificado"
print("\nNIVEL value_counts:")
print(df["NIVEL"].value_counts().to_string())

# Basura HTML
htmlish = df.apply(lambda s: s.astype(str).str.contains(r"<[^>]+>|nbsp|__VIEWSTATE", case=False, regex=True).sum())
print("\nCeldas con patrón HTML/viewstate:", int(htmlish.sum()))


### Conclusión 

Lista de problemas identificados (para alimentar el plan de limpieza):

1. **Faltantes altos en `DIRECTOR` (~29% NaN) y `TELEFONO` (~29% NaN)** — campos de contacto incompletos en origen.
2. **Faltantes disfrazados** en `DIRECTOR`/`TELEFONO`/`DIRECCION` (`S/D`, `.`, guiones de largo variable) que `isnull()` no ve → hay que normalizar a NaN.
3. **`DISTRITO` y `SUPERVISOR` co-ausentes** (~10,694 filas): patrón estructural, no aleatorio.
4. **`DISTRITO` con formatos heterogéneos** (`AA-BB-CCCC`, `AA-BBB`, e incompletos `##-`) → no asumir un solo regex en validaciones.
5. **`CIUDAD CAPITAL` vs `GUATEMALA`**: misma geografía administrativa partida en el dropdown; `MUNICIPIO` usa zonas solo en capital; `DEPARTAMENTAL` reparte ambas en NORTE/SUR/ORIENTE/OCCIDENTE. Decisión pendiente: fusionar o no.
6. **Tildes inconsistentes** entre `DEPARTAMENTO` (sin) y `DEPARTAMENTAL` (con) — no unir por string equality sin normalizar.
7. **Espacios dobles** en `ESTABLECIMIENTO` (4,676) — normalización de formato permitida; **no** corregir ortografía del nombre.
8. **`TELEFONO` multi-valor** (268) — una celda ≠ un teléfono; definir si se parte, se deja, o se toma el primero.
9. **Niveles `UNIVERSIDAD` (1) y `ADMINISTRATIVOS` (2)** fuera del criterio pedagógico “hasta diversificado” — decidir filtro.
10. **Sin duplicados de fila ni de `CODIGO`** — buena noticia; no hace falta deduplicar por llave.
11. **Sin basura HTML** en celdas — el parseo del scraper se ve limpio.
12. **Mismo nombre+dirección con varios códigos** (9,530) — esperado por niveles/jornadas; no borrar automáticamente.

Con esto cierra la **Parte 1 — Diagnóstico**. Siguiente: **Parte 2 — Plan de limpieza** (regla por variable, porqué y riesgos).
